# Benchmarking models for oxygen fugacity related variables

This notebook benchmarks the models for oxygen fugacity related variables in VolFe where possible.

## Python set-up

In [7]:
import VolFe as vf
import math
import pandas as pd
import Thermobar as tb

## Models for FMQ

option = FMQbuffer, function = FMQ

### 'ONeill87' O'Neill (1987) AmMin 72(1-2):67-75

Appendix A. Supplementary material - Supplementary data 2. Tab = Table S6 S redox calculator (sulfate,KSOg2=ONeill22.xlsx).

Cell R12: 0.13 [DFMQ]

Cell S12: -8.29 [logfO2]

Matches to 2 decimal places. Note spreadsheet uses +273 to convert to K, rather than 273.15 used in VolFe so T in spreadsheet = 1200.15 'C


In [8]:
PT = {"P":1000.}
PT["T"]=1200.
D = 0.13 # DFMQ

my_models = [["FMQbuffer", "ONeill87"]]
my_models = vf.make_df_and_add_model_defaults(my_models)

math.log10(vf.Dbuffer2fO2(PT,D,'FMQ',my_models))

-8.294378372874451

### 'Frost91' Frost (1991) in "Oxide Minerals: Petrologic and Magnetic Significance"

Comparison with results from Thermobar (Wieser et al., 2022) - values match perfectly.

Define input conditions:

In [28]:
PT = {"P":1000.}
PT["T"]=1200.
D = 0.13 # DFMQ

Run using VolFe

In [29]:
my_models = [["FMQbuffer", "Frost91"]]
my_models = vf.make_df_and_add_model_defaults(my_models)

math.log10(vf.Dbuffer2fO2(PT,D,'FMQ',my_models))

-8.096212368054847

Run using Thermobar

In [30]:
tb.calculate_logfo2_from_buffer_pos(T_K=PT["T"]+273.15,P_kbar=PT['P']/1000.,fo2_offset=D,buffer='QFM')

-8.096212368054847

## Models for NNO

option = NNObuffer, function = NNO

### 'Frost91' Frost (1991) in "Oxide Minerals: Petrologic and Magnetic Significance"

Comparison with results from Thermobar (Wieser et al., 2022) - values match perfectly.

Define input conditions:

In [25]:
PT = {"P":1000.}
PT["T"]=1200.
D = 0.13 # NNO

Run using VolFe

In [26]:
my_models = [["NNObuffer", "Frost91"]]
my_models = vf.make_df_and_add_model_defaults(my_models)

math.log10(vf.Dbuffer2fO2(PT,D,'NNO',my_models))

-7.401725893493535

Run using Thermobar

In [27]:
tb.calculate_logfo2_from_buffer_pos(T_K=PT["T"]+273.15,P_kbar=PT['P']/1000.,fo2_offset=D,buffer='NNO')

-7.401725893493535

## Models for fO2-Fe3+/FeT

option = fO2, function = fO2


In [32]:
my_analysis = {
        "Sample": "Hawaiian basalt",
        "SiO2": 51.29,  # wt%
        "TiO2": 2.50,  # wt%
        "Al2O3": 13.70,  # wt%
        "FeOT": 11.04,  # wt%
        "MnO": 0.02,  # wt%
        "MgO": 6.70,  # wt%
        "CaO": 11.03,  # wt%
        "Na2O": 2.27,  # wt%
        "K2O": 0.43,  # wt%
        "P2O5": 0.,  # wt%
        "H2O": 3.,  # wt%
        "CO2ppm": 1000.,  # ppm
        "STppm": 0.,  # ppm
        "Xppm": 0.0,  # ppm
        "Fe3FeT": 0.1,
    }

my_analysis = pd.DataFrame(my_analysis, index=[0])

PT = {"P":1000.}
PT["T"]=1200.

melt_wf=vf.melt_comp(0.,my_analysis)
melt_wf['CO2'] = my_analysis.loc[0.,"CO2ppm"]/1000000.
melt_wf["H2OT"] = my_analysis.loc[0,"H2O"]/100.
melt_wf["H2O"] = my_analysis.loc[0,"H2O"]/100.
melt_wf['ST'] = my_analysis.loc[0.,"STppm"]/1000000.
melt_wf['CT'] = (melt_wf['CO2']/vf.species.loc['CO2','M'])*vf.species.loc['C','M']
melt_wf['HT'] = (melt_wf['H2OT']/vf.species.loc['H2O','M'])*(2.*vf.species.loc['H','M'])
melt_wf['XT'] = my_analysis.loc[0.,"Xppm"]/1000000.
melt_wf["Fe3FeT"] = my_analysis.loc[0.,"Fe3FeT"]

In [33]:
my_models = [['fO2','Kress91']]
my_models = vf.make_df_and_add_model_defaults(my_models)

math.log10(vf.f_O2(PT,melt_wf,models=my_models))

-9.2695394895766

In [31]:
liq_comps=vf.melt_pysulfsat(melt_wf)
tb.convert_fe_partition_to_fo2(liq_comps=liq_comps, T_K=PT['T']+273.15, P_kbar=PT['P']/1000., model="Kress1991", Fe3Fet_Liq=melt_wf["Fe3FeT"],renorm=False)

overwriting Fe3Fet_Liq to that specified in the function input


AttributeError: 'float' object has no attribute 'astype'

Digitised two points from Fig. 3 from Kress & Carmichael (1991) using https://plotdigitizer.com/app (figure = fO2=Kress91,Kress91A [Fig3_Kress91].png), which are for mid-ocean ridge basalt JDFD-2.

P (GPa), T ('C), log10(fO2)
0.1000, 1351, -7.06
2.9168, 1393, -4.53
2.9168, 1393, -4.49

In [47]:
my_analysis = {
        "Sample": "JDFD2", # Table 1 from Carmichael (1991) CMP 106:129-141 doi.org/10.1007/BF00306429
        "SiO2": 50.62,  # wt%
        "TiO2": 1.96,  # wt%
        "Al2O3": 13.76,  # wt%
        'Fe2O3': 1.14, # wt%
        "FeO": 11.11,  # wt%
        "MnO": 0.22,  # wt%
        "MgO": 6.60,  # wt%
        "CaO": 10.74,  # wt%
        "Na2O": 2.76,  # wt%
        "K2O": 0.19,  # wt%
        "P2O5": 0.24,  # wt%
        "H2O": 0.,  # wt%
        "CO2ppm": 0.,  # ppm
        "STppm": 0.,  # ppm
        "Xppm": 0.0  # ppm
    }

my_analysis = pd.DataFrame(my_analysis, index=[0])

melt_wf=vf.melt_comp(0.,my_analysis)
melt_wf['CO2'] = my_analysis.loc[0.,"CO2ppm"]/1000000.
melt_wf["H2OT"] = my_analysis.loc[0,"H2O"]/100.
melt_wf["H2O"] = my_analysis.loc[0,"H2O"]/100.
melt_wf['ST'] = my_analysis.loc[0.,"STppm"]/1000000.
melt_wf['CT'] = (melt_wf['CO2']/vf.species.loc['CO2','M'])*vf.species.loc['C','M']
melt_wf['HT'] = (melt_wf['H2OT']/vf.species.loc['H2O','M'])*(2.*vf.species.loc['H','M'])
melt_wf['XT'] = my_analysis.loc[0.,"Xppm"]/1000000.
melt_wf["Fe2O3"] = my_analysis.loc[0.,"Fe2O3"]/100.
melt_wf["FeO"] = my_analysis.loc[0.,"FeO"]/100.
melt_wf["Fe3FeT"] = ((2.0*melt_wf["Fe2O3"])/vf.species.loc["Fe2O3","M"])/(((2.0*melt_wf["Fe2O3"])/vf.species.loc["Fe2O3","M"]) + (melt_wf["FeO"]/vf.species.loc["FeO","M"]))

In [51]:
PT = {"P":0.1000*10000}
PT["T"]=1351.

In [52]:
my_models = [['fO2','Kress91']]
my_models = vf.make_df_and_add_model_defaults(my_models)

math.log10(vf.f_O2(PT,melt_wf,models=my_models))

-8.518429412310038

In [50]:
my_models = [['fO2','Kress91A']]
my_models = vf.make_df_and_add_model_defaults(my_models)

math.log10(vf.f_O2(PT,melt_wf,models=my_models))

-8.172841850581397

In [53]:
PT = {"P":2.9168*10000}
PT["T"]=1393.

In [54]:
my_models = [['fO2','Kress91']]
my_models = vf.make_df_and_add_model_defaults(my_models)

math.log10(vf.f_O2(PT,melt_wf,models=my_models))

-5.934567167125134

In [55]:
my_models = [['fO2','Kress91A']]
my_models = vf.make_df_and_add_model_defaults(my_models)

math.log10(vf.f_O2(PT,melt_wf,models=my_models))

-5.62867347870724

### 'ONeill18' Eq. (9a) in O'Neill et al. (2018) EPSL 504:152-162

Appendix A. Supplementary material - Supplementary data 2. Tab = Table S6 S redox calculator (sulfate,KSOg2=ONeill22.xlsx).

Cell S12: -8.29 [logfO2]

Matches to 2 decimal places. Note spreadsheet uses +273 to convert to K, rather than 273.15 used in VolFe so T in spreadsheet = 1200.15 'C

In [4]:
my_models = [['fO2','ONeill18']]
my_models = vf.make_df_and_add_model_defaults(my_models)

math.log10(vf.f_O2(PT,melt_wf,models=my_models))

-8.289923385626128